In [ ]:
import pickle as pkl
import numpy as np

data1 = pkl.load(open('dupes_1_sleep_0.05.pkl', 'rb'))
data2 = pkl.load(open('dupes_2_sleep_0.05.pkl', 'rb'))
data3 = pkl.load(open('dupes_3_sleep_0.05.pkl', 'rb'))
data4 = pkl.load(open('dupes_4_sleep_0.05.pkl', 'rb'))
data5 = pkl.load(open('dupes_5_sleep_0.05.pkl', 'rb'))
data6 = pkl.load(open('dupes_6_sleep_0.05.pkl', 'rb'))
data7 = pkl.load(open('dupes_7_sleep_0.05.pkl', 'rb'))


all_data = data1 + data2 + data3 + data4 + data5 + data6 + data7
print(data1[10])

# Store pairs where sensor data are different: (state, action, next_state, delta_t)
delta_t = []
training_tuples = []

for i in range(1, len(all_data) - 1):
    timestamp_0, sensor_data_0, mocap_data_0 = all_data[i-1]
    timestamp_1, sensor_data_1, mocap_data_1 = all_data[i]
    timestamp_2, sensor_data_2, mocap_data_2 = all_data[i + 1]
    # print(sensor_data_0)
    # Check if any sensor_data or mocap_data contain NaN values
    if (np.any(np.isnan(sensor_data_0)) or np.any(np.isnan(sensor_data_1)) or np.any(np.isnan(sensor_data_2)) or
        np.any(np.isnan(mocap_data_0)) or np.any(np.isnan(mocap_data_1)) or np.any(np.isnan(mocap_data_2))):
        continue
    
    # Calculate time differences
    delta_t_1 = timestamp_1 - timestamp_0
    delta_t_2 = timestamp_2 - timestamp_1
    
    # Check if sensor data are different
    if not np.array_equal(sensor_data_1, sensor_data_2):
        # Calculate velocities correctly
        # Velocity for state_1: from mocap_data_0 to mocap_data_1
        xyz_vel_1 = (mocap_data_1[:3] - mocap_data_0[:3]) / delta_t_1
        # Velocity for state_2: from mocap_data_1 to mocap_data_2
        xyz_vel_2 = (mocap_data_2[:3] - mocap_data_1[:3]) / delta_t_2
        
        # sensor_array: [height, pitch_body, - roll_body, yaw_body, pitch rate, - roll rate, yaw rate, y acceleration, - x acceleration, z acceleration, motor_1, motor_2, servo_angle_deg]
        # mocap_array: [pos_x, pos_y, pos_z, rot_roll(abt global y), rot_pitch(abt global x), rot_yaw(about global z)]
        # state = x_vel, y_vel, z_vel, height, roll_body, pitch_body, yaw_body, roll rate, pitch rate, yaw rate
        state_1 = xyz_vel_1.tolist() + [-sensor_data_1[2], sensor_data_1[1], -sensor_data_1[3], -sensor_data_1[5], sensor_data_1[4], sensor_data_1[6]] 
        action = [sensor_data_1[10], sensor_data_1[11], np.deg2rad(sensor_data_1[12])]
        state_2 = xyz_vel_2.tolist() + [-sensor_data_2[2], sensor_data_2[1], -sensor_data_2[3], -sensor_data_2[5], sensor_data_2[4], sensor_data_2[6]]
        
        # Store: (state, action, next_state, delta_t)
        training_tuples.append((np.array(state_1), np.array(action), np.array(state_2)))
        delta_t.append(delta_t_2)

print(f"Total pairs with different sensor data: {len(training_tuples)}")
print(f"Total data points: {len(all_data)}")
if len(delta_t) > 0:
    print("Average time between data points: ", np.mean(delta_t))
    print("standard deviation of time between data points: ", np.std(delta_t))
else:
    print("No training tuples created")

(0.6370890140533447, array(nan, dtype=float32), array([ -2.0686955,  -1.9901108,   0.8163249,  -4.9139805,   4.752332 ,
       -15.951497 ], dtype=float32))
Total pairs with different sensor data: 49167
Total data points: 75090
Average time between data points:  0.050216204193848896
standard deviation of time between data points:  0.004182072744490542


In [29]:
def training_data_to_tuples(all_data):
    training_tuples = []
    delta_t = []
    for i in range(1, len(all_data) - 1):
        timestamp_0, sensor_data_0, mocap_data_0 = all_data[i-1]
        timestamp_1, sensor_data_1, mocap_data_1 = all_data[i]
        timestamp_2, sensor_data_2, mocap_data_2 = all_data[i + 1]
        # print(sensor_data_0)
        # Check if any sensor_data or mocap_data contain NaN values
        if (np.any(np.isnan(sensor_data_0)) or np.any(np.isnan(sensor_data_1)) or np.any(np.isnan(sensor_data_2)) or
            np.any(np.isnan(mocap_data_0)) or np.any(np.isnan(mocap_data_1)) or np.any(np.isnan(mocap_data_2))):
            continue
    
        # Calculate time differences
        delta_t_1 = timestamp_1 - timestamp_0
        delta_t_2 = timestamp_2 - timestamp_1
        
        # Check if sensor data are different
        if not np.array_equal(sensor_data_1, sensor_data_2):
            # Calculate velocities correctly
            # Velocity for state_1: from mocap_data_0 to mocap_data_1
            xyz_vel_1 = (mocap_data_1[:3] - mocap_data_0[:3]) / delta_t_1
            # Velocity for state_2: from mocap_data_1 to mocap_data_2
            xyz_vel_2 = (mocap_data_2[:3] - mocap_data_1[:3]) / delta_t_2
            
            # sensor_array: [height, pitch_body, - roll_body, yaw_body, pitch rate, - roll rate, yaw rate, y acceleration, - x acceleration, z acceleration, motor_1, motor_2, servo_angle_deg]
            # mocap_array: [pos_x, pos_y, pos_z, rot_roll(abt global y), rot_pitch(abt global x), rot_yaw(about global z)]
            # state = x_vel, y_vel, z_vel, height, roll_body, pitch_body, yaw_body, roll rate, pitch rate, yaw rate
            state_1 = xyz_vel_1.tolist() + [-sensor_data_1[2], sensor_data_1[1], -sensor_data_1[3], -sensor_data_1[5], sensor_data_1[4], sensor_data_1[6]] 
            action = [sensor_data_1[10], sensor_data_1[11], np.deg2rad(sensor_data_1[12])]
            state_2 = xyz_vel_2.tolist() + [-sensor_data_2[2], sensor_data_2[1], -sensor_data_2[3], -sensor_data_2[5], sensor_data_2[4], sensor_data_2[6]]
            
            # Store: (state, action, next_state, delta_t)
            training_tuples.append((np.array(state_1), np.array(action), np.array(state_2)))
            delta_t.append(delta_t_2)

    print(f"Total pairs with different sensor data: {len(training_tuples)}")
    print(f"Total data points: {len(all_data)}")
    if len(delta_t) > 0:
        print("Average time between data points: ", np.mean(delta_t))
        print("standard deviation of time between data points: ", np.std(delta_t))
    else:
        print("No training tuples created")

    return training_tuples

In [31]:
dupe_8 = pkl.load(open('dupes_8_sleep_0.05.pkl', 'rb'))
dupe_9 = pkl.load(open('dupes_9_ag1.pkl', 'rb'))
dupe_8_pk = training_data_to_tuples(dupe_8)
dupe_9_pk = training_data_to_tuples(dupe_9)
pkl.dump(dupe_8_pk, open('dupe_8.pkl', 'wb'))
pkl.dump(dupe_9_pk, open('dupe_9_ag.pkl', 'wb'))

print(dupe_8_pk[55])

Total pairs with different sensor data: 6915
Total data points: 10709
Average time between data points:  0.048602194624022994
standard deviation of time between data points:  0.0040968699940354585
Total pairs with different sensor data: 4894
Total data points: 7411
Average time between data points:  0.0488413891891679
standard deviation of time between data points:  0.004140760316967532
(array([ 0.12086585, -0.19507706,  0.22863436, -0.11337952, -0.24798664,
        0.50903898, -0.0258336 , -0.12329572,  0.40155178]), array([0.771503 , 0.3422309, 1.3542606], dtype=float32), array([ 0.12131342, -0.19240773,  0.20987694, -0.11996427, -0.25109076,
        0.48638299, -0.02053854, -0.11769988,  0.48989123]))


In [27]:
print(len(training_tuples))
pkl.dump(training_tuples, open('real_training_tuples_75K_0_05.pkl', 'wb'))

49167


In [ ]:
# Convert training_data format (state, action, next_state) to batch format

import matplotlib.pyplot as plt
import numpy as np
import torch

# ----- Offline training (supervised next-step regression) -----
batch_x_list = []
batch_y_list = []


for state, action, next_state in training_data:
    obs_tensor = torch.from_numpy(state).float()
    action_tensor = torch.from_numpy(action).float()
    next_obs_tensor = torch.from_numpy(next_state).float()
    
    batch_x_list.append(torch.cat([obs_tensor, action_tensor], dim=-1))
    batch_y_list.append(next_obs_tensor)

# Convert to tensors with correct dimensions
batch_x = torch.stack(batch_x_list, dim=0)  # shape: [B, 12] (9 state + 3 action)
batch_y = torch.stack(batch_y_list, dim=0)


In [ ]:
# Plot distributions of all features in batch_x

# Get number of features
num_features = batch_x.shape[1]

# Feature names (9 state features + 3 action features)
feature_names = [
    'x_vel', 'y_vel', 'z_vel',  # velocities
    'roll', 'pitch', 'yaw',     # angles (or angular velocities)
    'roll_vel', 'pitch_vel', 'yaw_vel',  # angular velocities
    'left_motor', 'right_motor', 'servo'  # actions
]

# Extract all features for plotting
features = [batch_x[:, i].cpu().numpy() for i in range(num_features)]

# Create subplots for histograms (3 rows x 4 columns for 12 features)
n_cols = 4
n_rows = (num_features + n_cols - 1) // n_cols  # Ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))

# Flatten axes for easier indexing
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes_flat = axes.flatten()

# Plot histograms for all features
colors = plt.cm.tab20(np.linspace(0, 1, num_features))
for i in range(num_features):
    ax = axes_flat[i]
    ax.hist(features[i], bins=50, alpha=0.7, color=colors[i], edgecolor='black')
    ax.set_title(f'batch_x[{i}] ({feature_names[i] if i < len(feature_names) else f"feature_{i}"})')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for i in range(num_features, len(axes_flat)):
    axes_flat[i].axis('off')

plt.tight_layout()
plt.show()

# Create subplots for box plots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes_flat = axes.flatten()

# Plot box plots for all features
for i in range(num_features):
    ax = axes_flat[i]
    feat_name = feature_names[i] if i < len(feature_names) else f"feature_{i}"
    bp = ax.boxplot(features[i], patch_artist=True, 
                    boxprops=dict(facecolor=colors[i], alpha=0.7))
    ax.set_title(f'Box Plot: x[{i}] = {feat_name}')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for i in range(num_features, len(axes_flat)):
    axes_flat[i].axis('off')

plt.tight_layout()
plt.show()

# Print statistics for all features
print("Statistics for batch_x features:")
for i in range(num_features):
    feat_name = feature_names[i] if i < len(feature_names) else f"feature_{i}"
    print(f"x[{i}] = {feat_name} - Mean: {np.mean(features[i]):.4f}, "
          f"Std: {np.std(features[i]):.4f}, Min: {np.min(features[i]):.4f}, "
          f"Max: {np.max(features[i]):.4f}")

# Combined distribution plot for all features
plt.figure(figsize=(14, 8))
for i in range(num_features):
    feat_name = feature_names[i] if i < len(feature_names) else f"feature_{i}"
    plt.hist(features[i], bins=50, alpha=0.4, 
             label=f'x[{i}] = {feat_name}', color=colors[i])
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Combined Distribution of All batch_x Features')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
